*   A base classification model collects shared behavior for classifiers, especially prediction and accuracy measurement.
*   This notebook builds those ideas manually before using class structure.

In [4]:
import math
import random
import numpy as np
import torch

# 4.3.1 The Classifier Class

*   A classifier is a model that predicts class labels.

*   A base class is a parent class that stores behavior shared by several related classes.
> *   e.g. `class Animal` has `def eat(self)` and `def sleep(self)`
*   In Python, inheritance lets a child class reuse parent behavior
> * e.g. `class Dog(Animal)`

## 2. Why this exists

*   Many classifiers share the same evaluation logic even when their internal model architecture differs.
*   A base class avoids rewriting that shared code.

## 3. Examples

*   Plain Python classifier interface.

In [5]:
class SimpleClassifier:
  def predict(self, logits):
    return torch.argmax(logits, dim=1) # torch.argmax() returns the index of the largest value.

model = SimpleClassifier()
logits = torch.tensor([[1.0, 3.0], [2.0, 0.5]])
model.predict(logits) # [1, 0] because 3 and 2 are the largest values inside logits

tensor([1, 0])

## 4. Step-by-step breakdown

*   `class SimpleClassifier:` defines a class.

*   `predict` is a method that receives logits.

*   `torch.argmax(logits, dim=1)` selects the class index with the largest score for each row.

*   `model.predict(logits)` returns predicted class IDs.

## 5. Connection to ML systems

*   D2L and PyTorch code often define classifier classes so training, validation, and prediction can use a common interface.

## 6. Common confusion points

- A classifier predicts categories, not continuous values.
- `argmax` chooses the index of the largest value.
- `dim=1` means compare across class columns.
- A base class should contain shared behavior, not task-specific hidden assumptions.

# 4.3.2 Accuracy

## 1. Intuition

*   Accuracy is the fraction of predictions that match the correct labels.

*   A correct prediction is one where predicted class ID equals the true class ID.

## 2. Why this exists

*   Classification loss measures probability quality, while accuracy measures final decision correctness.
*   Both are useful but answer different questions.

## 3. Examples

*   Manual accuracy computation.

In [6]:
preds = torch.tensor([1, 0, 1, 1])
labels = torch.tensor([1, 1, 1, 0])
correct = (preds == labels) # [True, False, True, False]
accuracy = correct.float().mean() # (1+0+1+0) / 4 = 2/4 = 0.5
accuracy

tensor(0.5000)

*   Accuracy from logits.

In [8]:
logits = torch.tensor([[0.1, 0.9], [2.0, 1.0]])
labels = torch.tensor([1, 0])
preds = torch.argmax(logits, dim=1) # Returns [1, 0] since 0.9 and 2 are the largest values
(preds == labels).float().mean() # 1==1 and 0==0, (1+1)/2 = 1

tensor(1.)

## 4. Step-by-step breakdown

*   `preds == labels` creates Boolean values.

*   `True` means correct and `False` means incorrect.

*   `.float()` converts `True` to 1.0 and `False` to 0.0.

*   `.mean()` averages those 0/1 values into a fraction correct.

## 5. Connection to ML systems

*   Training dashboards often report both loss and accuracy.
*   Loss can improve before accuracy changes because probabilities can improve without changing the final class choice.
> * Accuracy only cares whether the predicted class is correct.
> * Loss cares about how confident the model is in its prediction.
> * A model can become more confident in the correct answer (lowering the loss) without changing which class it predicts (so accuracy stays the same).

## 6. Common confusion points

- Accuracy ignores confidence. It's a Boolean (True or False).
- Accuracy can be misleading on imbalanced datasets.
- Use `argmax` on logits or probabilities; both preserve score ordering.
- Accuracy is not differentiable, so it is usually not the training loss.

    ## Why can't we train using accuracy?

    Imagine a binary classifier where the **true label is class 1**.

    | Probability of Class 1 | Prediction | Accuracy |
    |:----------------------:|:----------:|:--------:|
    | 0.51 | Correct | 1 |
    | 0.60 | Correct | 1 |
    | 0.90 | Correct | 1 |

    Although the model becomes **more confident** in the correct class (from 51% to 90%), the **accuracy remains exactly the same** because the predicted class never changes.

    > **Accuracy only measures whether the predicted class is correct, not how confident the prediction is.**
    
    From the optimizer's perspective, there is no difference between a prediction that is **51% confident** and one that is **99% confident**—both receive an accuracy of **1**. As a result, accuracy provides no information about how to improve the model further.

    In contrast, **cross-entropy loss** rewards increasing confidence in the correct class:

    | Probability of Class 1 | Accuracy | Cross-Entropy Loss |
    |:----------------------:|:--------:|:------------------:|
    | 0.51 | 1 | Higher |
    | 0.60 | 1 | Lower |
    | 0.90 | 1 | Much Lower |

    As the probability assigned to the correct class increases, the **loss decreases smoothly**, giving the optimizer a continuous signal for updating the model's parameters.

    **Key idea:**  
    - **Accuracy** is a **performance metric** used to evaluate the model.
    - **Cross-entropy loss** is the **training objective** because it is differentiable and provides meaningful gradients for optimization.

# 4.3.3 Summary

## 1. Intuition

*   A classifier maps examples to class scores, predicted class IDs, and evaluation metrics.

*   Accuracy is the simplest classification metric.

## 2. Why this exists

*   Separating prediction logic from metric logic makes training code easier to reuse.

## 3. Examples

*   Package prediction and accuracy together.

In [9]:
def accuracy_from_logits(logits, labels):
  preds = torch.argmax(logits, dim=1)
  return (preds == labels).float().mean()

accuracy_from_logits(logits, labels)

tensor(1.)

## 4. Step-by-step breakdown

*   The helper computes predicted class IDs from logits.

*   It compares predictions against labels.

*   It returns the average correctness as a scalar tensor.

## 5. Connection to ML systems

*   This helper is a tiny version of the evaluation code used by classifier classes.

## 6. Common confusion points

- Prediction and evaluation are related but separate steps.
- Accuracy returns an estimate over the examples provided.
- Larger evaluation sets usually give more stable estimates.
- Metric helpers should document expected shapes.

# 4.3.4 Exercises

## 1. Intuition

*   These exercises test whether you can compute classifier predictions and accuracy from logits.

## 2. Why this exists

*   Accuracy is simple, but shape mistakes can still produce wrong results.

## 3. Examples

*   Exercise 1: predict class IDs for three examples.

In [10]:
logits = torch.tensor([[1.0, 0.0], [0.2, 0.8], [3.0, 4.0]])
preds = torch.argmax(logits, dim=1) # [0, 1, 1]
preds

tensor([0, 1, 1])

*   Exercise 2: compute accuracy.

In [11]:
labels = torch.tensor([0, 1, 0])
acc = (preds == labels).float().mean() # [True, True, False] = (1+1+0) / 3 = 2/3 = 0.667
acc

tensor(0.6667)

## 4. Step-by-step breakdown

*   Exercise 1 checks `argmax` across the class dimension.

*   Exercise 2 checks Boolean comparison and averaging.

*   The third example is wrong because class 1 has the larger score while the label is 0.

## 5. Connection to ML systems

*   These exact operations appear in validation loops for softmax regression and later classifiers.

## 6. Common confusion points

- Class IDs start at 0 in PyTorch examples.
- Prediction shape should match label shape.
- Accuracy treats all errors equally.
- Accuracy should be computed without updating model parameters.